# Early exploratory notebook (historical)

This notebook preserves exploratory code from an earlier version of this project. All saved outputs have been removed. The original firewall-log dataset is no longer available, so this notebook cannot be reproduced as-is. Several exploratory decisions were revised in the maintained `risk_pipeline.py`; see `docs/technical_review.md` for details.


In [ ]:
# Read the local CSV file
import pandas as pd
data_firewall = pd.read_csv("DATA/Firewall_Threat_20250401-20250410.csv")

In [ ]:
# Define the work periods by day, hour and the resulting Period Threat

def assign_period_day(day):
    if 5 <= day <= 6 :      # 0 = Monday, 6 = Sunday
        return 'Unusual'
    else:
        return 'Normal'

def assign_period_hour(hour):
    if 9 <= hour < 12 :
        return 'Normal'
    elif 12 <= hour < 13 :
        return 'Lunch'
    elif 13 <= hour < 18 :
        return 'Normal'
    elif 18 <= hour < 23 :
        return 'Unusual evening'
    elif 7 <= hour < 9 :
        return 'Unusual morning'
    else:
        return 'Unusual night'
    
def assign_threat_period(work_day,work_hour):
    if work_day == 'Unsual' :
        return 1
    # elif work_hour in ['Lunch', 'Unusual evening', 'Unusual morning', 'Unusual night'] :
        # return 1
    elif work_hour == 'Unusual night' :
        return 1
    elif work_hour == 'Unusual evening' :
        return 0.8
    elif work_hour == 'Unusual morning' :
        return 0.6
    elif work_hour == 'Lunch' :
        return 0.4
    else :
        return 0

In [ ]:
# Define the amount of transfered data and the resulting Bytes Threat

def assign_threat_bytes(bytes):
    try:
        bytes = int(bytes)  # Attempt conversion
        if 0 <= bytes < 50  :
            return 0
        elif 50 <= bytes < 200  :
            return 0.6
        elif 200 <= bytes < 500  :
            return 0.8
        else :
            return 1
    except ValueError:  # Handle conversion errors
        return None  # Or return a default value

In [ ]:
# Define the Global Threat resulting of the Period-Bytes Threats

def assign_threat_global(threat_period, threat_bytes):
    if threat_period != 0 and threat_bytes != 0 :
        return ( threat_period + threat_bytes ) / 2
    elif threat_period == 0 and threat_bytes != 0 :
        return threat_bytes
    elif threat_period != 0 and threat_bytes == 0 :
        return threat_period
    else :
        return 0

In [ ]:
# Define the Localtion Threat from the 'geo_src_ip_country'

def assign_threat_location(geo_src_ip_country):
    if geo_src_ip_country == 'tw' :
        return 0
    else :
        return 1

In [ ]:
# STEP 1 : CONVERT TIME TO DATETIME AND CREAT DAY AND HOUR COLUMNS
import pandas as pd
import numpy as np
# Convert 'time' to datetime
data_firewall['DATE_TIME'] = pd.to_datetime(data_firewall['DATE'] + ' ' + data_firewall['TIME'], format="%Y-%m-%d %H:%M:%S")

# Extract Day of the Week and Hour
data_firewall['HOUR'] = data_firewall['DATE_TIME'].dt.hour
data_firewall['DAY_OF_WEEK'] = data_firewall['DATE_TIME'].dt.dayofweek  # 0 = Monday, 6 = Sunday

# Display a sample of the dataset
data_firewall[['DATE_TIME','DAY_OF_WEEK', 'HOUR']].head()

In [ ]:
# APPLY THE WORK PERIODS AND THE THREATS THRESHOLDS

data_firewall['WORK_PERIOD_DAY'] = data_firewall['DAY_OF_WEEK'].apply(assign_period_day)
data_firewall['WORK_PERIOD_HOUR'] = data_firewall['HOUR'].apply(assign_period_hour)
data_firewall['THREAT_PERIOD'] = data_firewall.apply(lambda row: assign_threat_period(row['WORK_PERIOD_DAY'], row['WORK_PERIOD_HOUR']), axis=1)

data_firewall['THREAT_BYTES'] = data_firewall['RCVD_BYTE'].apply(assign_threat_bytes)

data_firewall['THREAT_GLOBAL'] = data_firewall.apply(lambda row: assign_threat_global(row['THREAT_PERIOD'], row['THREAT_BYTES']), axis=1)

# Check the new column
data_firewall.head()

In [ ]:
# STEP 3 : FIND OUT THE NUMBER OF USERS INSIDE THE THREATS THRESHOLDS
from IPython.display import display

# Cmpute the number of threat users
frwl_threat_users_period = data_firewall.groupby(['THREAT_PERIOD']).size().reset_index(name='user')
frwl_threat_users_bytes = data_firewall.groupby(['THREAT_BYTES']).size().reset_index(name='user')
frwl_threat_users_global = data_firewall.groupby(['THREAT_GLOBAL']).size().reset_index(name='user')

frwl_total_threat_users_period = frwl_threat_users_period['user'].sum()
frwl_total_threat_users_bytes = frwl_threat_users_bytes['user'].sum()
frwl_total_threat_users_global = frwl_threat_users_global['user'].sum()

# Add percentage column
frwl_threat_users_period['percentage'] = round((frwl_threat_users_period['user'] / frwl_total_threat_users_period) * 100 , 2)
frwl_threat_users_bytes['percentage'] = round((frwl_threat_users_bytes['user'] / frwl_total_threat_users_bytes) * 100 , 2)
frwl_threat_users_global['percentage'] = round((frwl_threat_users_global['user'] / frwl_total_threat_users_global) * 100 , 2)

# Display the results
frwl_threat_users_period.columns = ['Frwl_Threat_Period', 'User Count', 'User Percentage']
frwl_threat_users_bytes.columns = ['Frwl_Threat_Bytes', 'User Count', 'User Percentage']
frwl_threat_users_global.columns = ['Frwl_Threat_Global', 'User Count', 'User Percentage']

display(frwl_threat_users_global, frwl_threat_users_period, frwl_threat_users_bytes)

In [ ]:
# Save the updated data files as local CSV files
data_firewall.to_csv("Firewall_Threat_limpio1.csv", index= False)
print("Files have been saved as a local csv file.")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler, MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns

# Load cleaned firewall log
df = pd.read_csv("Firewall_Threat_limpio1.csv", parse_dates=["DATE_TIME"])

# Convert relevant columns to numeric
df['RCVD_BYTE'] = pd.to_numeric(df['RCVD_BYTE'], errors='coerce')
df['SENT_BYTE'] = pd.to_numeric(df['SENT_BYTE'], errors='coerce')
df['DURATION'] = pd.to_numeric(df['DURATION'], errors='coerce')
df['DESTINATION_PORT'] = pd.to_numeric(df['DESTINATION_PORT'], errors='coerce')

# Feature Engineering
df['BYTE_RATIO'] = df['SENT_BYTE'] / (df['RCVD_BYTE'] + 1)
df['BYTE_RATIO'] = df['BYTE_RATIO'].clip(upper=10)  # Cap extreme values

# Create flag features
df['IS_DENY'] = df['ACTION'].apply(lambda x: 1 if str(x).lower() == 'deny' else 0)
df['IS_HIGH_PORT'] = df['DESTINATION_PORT'].apply(lambda x: 1 if x >= 1024 else 0)

# Normalize BYTE_RATIO
df['BYTE_RATIO_NORM'] = MinMaxScaler().fit_transform(df[['BYTE_RATIO']])

# %%
# Select features and normalize for anomaly detection
features = ['THREAT_PERIOD', 'THREAT_BYTES', 'BYTE_RATIO', 'IS_DENY', 'IS_HIGH_PORT', 'DURATION']
X = df[features].copy().fillna(0)

X_scaled = RobustScaler().fit_transform(X)

# Isolation Forest for anomaly detection
iso = IsolationForest(n_estimators=100, contamination=0.05, random_state=0)
df['ISO_ANOMALY'] = iso.fit_predict(X_scaled)

# Convert -1 to 1 (anomaly), 1 to 0 (normal)
df['ANOMALY_BIN'] = df['ISO_ANOMALY'].apply(lambda x: 1 if x == -1 else 0)

# %%
# Calculate preliminary risk score
df['RISK_SCORE'] = (
    df['THREAT_PERIOD'] * 0.2 +
    df['THREAT_BYTES'] * 0.1 +
    df['BYTE_RATIO_NORM'] * 0.1 +
    df['IS_DENY'] * 0.1 +
    df['IS_HIGH_PORT'] * 0.1 +
    df['ANOMALY_BIN'] * 0.4  
    )


# Normalize final risk score to 0–1
df['RISK_SCORE'] = MinMaxScaler().fit_transform(df[['RISK_SCORE']])

# %%
# Aggregate per user (SOURCE_IP)
user_df = df.groupby('SOURCE_IP').agg({
    'RISK_SCORE': 'mean',
    'THREAT_PERIOD': 'mean',
    'THREAT_BYTES': 'mean',
    'BYTE_RATIO': 'mean',
    'DURATION': 'mean',
    'IS_DENY': 'mean',
    'ANOMALY_BIN': 'mean',
    'DATE_TIME': ['min', 'max', 'count']
}).reset_index()

# Flatten columns
user_df.columns = ['SOURCE_IP', 'RISK_SCORE', 'THREAT_PERIOD', 'THREAT_BYTES', 'BYTE_RATIO',
                   'DURATION', 'IS_DENY', 'ANOMALY_RATE', 'FIRST_SEEN', 'LAST_SEEN', 'NUM_EVENTS']

# Drop NaNs if any
user_df = user_df[user_df['RISK_SCORE'].notna()]

# %%
# Add risk level
def classify_risk(score):
    if score >= 0.7:
        return 'High'
    elif score >= 0.4:
        return 'Medium'
    return 'Low'

user_df['RISK_LEVEL'] = user_df['RISK_SCORE'].apply(classify_risk)

# Plot histogram of risk score
plt.figure(figsize=(10, 6))
sns.histplot(data=user_df, x='RISK_SCORE', bins=np.linspace(0, 1, 21), kde=True, color='skyblue', edgecolor='black')
plt.title('Distribution of User Risk Scores')
plt.xlabel('Risk Score (0.0 - 1.0)')
plt.ylabel('User Count')
plt.xticks(np.linspace(0, 1, 11))
plt.tight_layout()
plt.show()

# Plot count of users by risk level
plt.figure(figsize=(6, 4))
sns.countplot(data=user_df, x='RISK_LEVEL', order=['Low', 'Medium', 'High'], palette='coolwarm')
plt.title('User Risk Level Classification')
plt.xlabel('Risk Level')
plt.ylabel('User Count')
plt.tight_layout()
plt.show()


user_df.to_csv("final_user_risk_report.csv", index=False)


In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import RobustScaler
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

#  Log-transform skewed features to reduce scale distortion
user_df['LOG_THREAT_BYTES'] = np.log1p(user_df['THREAT_BYTES'])
user_df['LOG_DURATION'] = np.log1p(user_df['DURATION'])

#  Select final features for clustering 
clustering_features = user_df[['THREAT_PERIOD', 'LOG_THREAT_BYTES', 'BYTE_RATIO', 'LOG_DURATION', 'IS_DENY', 'ANOMALY_RATE']]

#  Normalize features
X_clustering = RobustScaler().fit_transform(clustering_features)

#  Apply KMeans clustering (k=4 for 4 roles)
kmeans = KMeans(n_clusters=4, random_state=42)
user_df['CLUSTER_ID'] = kmeans.fit_predict(X_clustering)

# Analyze clusters and assign role names
cluster_summary = user_df.groupby('CLUSTER_ID')[['THREAT_PERIOD', 'THREAT_BYTES', 'BYTE_RATIO', 'DURATION', 'IS_DENY', 'ANOMALY_RATE']].mean().round(2)
print("🔎 Cluster Behavior Summary:\n", cluster_summary)

#  Manually map cluster IDs to PDF roles 
# You may revise this after inspecting the printed summary!
cluster_roles = {
    0: 'Sales',
    1: 'Developer',
    2: 'Management',
    3: 'Data Analyst'
}
user_df['ROLE'] = user_df['CLUSTER_ID'].map(cluster_roles)

# === Step 7: Visualize clusters using PCA ===
X_pca = PCA(n_components=2).fit_transform(X_clustering)

plt.figure(figsize=(10, 6))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=user_df['ROLE'], palette='Set2', s=60)
plt.title('User Behavior Clustering (KMeans + PCA)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend(title='Inferred Role')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=user_df, x='ROLE', y='RISK_SCORE', palette='Set2')
plt.title('Risk Score Distribution by Inferred Role')
plt.xlabel('User Role')
plt.ylabel('Risk Score')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1) Pick the top‐risk SOURCE_IP
top_user = user_df.sort_values("RISK_SCORE", ascending=False).iloc[0]["SOURCE_IP"]
user_activity = df[df["SOURCE_IP"] == top_user].copy()

# 2) Compute that user’s first/last timestamp
user_start = user_activity["DATE_TIME"].min()
user_end   = user_activity["DATE_TIME"].max()

# 3) Add a small padding (e.g. ±10 minutes) so you see empty space around their events
padding = pd.Timedelta(minutes=10)
plot_start = user_start - padding
plot_end   = user_end + padding

# 4) Now draw the histogram, then override x‐limits
plt.figure(figsize=(10, 4))
sns.histplot(user_activity["DATE_TIME"], bins=30, color="crimson")

# 5) Force the x‐axis to run from (user_start – 10 minutes) to (user_end + 10 minutes)
plt.xlim(plot_start, plot_end)

plt.title(f"Access Timeline – {top_user}")
plt.xlabel("Time")
plt.ylabel("Number of Events")
plt.tight_layout()
plt.show()


In [ ]:
df['HOUR'] = df['DATE_TIME'].dt.hour
df['DAY'] = df['DATE_TIME'].dt.day_name()

# Define correct weekday order
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Pivot and reorder
heat_data = (
    df.pivot_table(index='DAY', columns='HOUR', values='SOURCE_IP', aggfunc='count')
      .reindex(day_order)
      .fillna(0)
)

plt.figure(figsize=(12, 6))
sns.heatmap(heat_data, cmap='YlOrRd')
plt.title("Access Frequency Heatmap (Day vs Hour)")
plt.xlabel("Hour of Day")
plt.ylabel("Day of Week")
plt.tight_layout()
plt.show()


In [ ]:
alert_table = user_df.sort_values('RISK_SCORE', ascending=False)[['SOURCE_IP', 'RISK_SCORE', 'ROLE', 'NUM_EVENTS', 'FIRST_SEEN', 'LAST_SEEN']].head(10)
print(alert_table)

In [ ]:
# 1) Print all anomaly rows:
anomalies = df[df['ANOMALY_BIN'] == 1]
print("Total anomalies:", len(anomalies))
print(anomalies)

# 2) If you just want to see a subset (e.g., first 20 anomalies):
print(anomalies.head(20))

# 3) Or display selected columns alongside ANOMALY_BIN:
cols_to_show = ['DATE_TIME', 'SOURCE_IP', 'THREAT_BYTES', 'DURATION', 'ANOMALY_BIN']
print(df.loc[df['ANOMALY_BIN'] == 1, cols_to_show].head(20))


In [ ]:
# Prepare a “date only” column
df["DATE_TIME"] = pd.to_datetime(df["DATE_TIME"], errors="coerce")
df["DATE_ONLY"] = df["DATE_TIME"].dt.floor("D")

# ──────────────────────────────────────────────────────────────────────────────
# Count total events & anomalies per day
daily_total   = df.groupby("DATE_ONLY").size().rename("total_events")
daily_anomaly = df.groupby("DATE_ONLY")["ANOMALY_BIN"].sum().rename("anomaly_count")
daily_summary  = pd.concat([daily_total, daily_anomaly], axis=1).fillna(0)
daily_summary["anomaly_rate"] = (daily_summary["anomaly_count"] / daily_summary["total_events"]) * 100.0


left_date  = pd.to_datetime("2025-03-31")
right_date = pd.to_datetime("2025-04-11")

# 2) Subset daily_summary to the desired interval [left_date, right_date]
mask         = (daily_summary.index > left_date) & (daily_summary.index < right_date)
ds_sub = daily_summary.loc[mask]

plt.figure(figsize=(10, 5))

# Total events as light gray bars
plt.bar(
    ds_sub.index,
    ds_sub["total_events"],
    color="lightgray",
    label="Total Events",
    alpha=0.6
)

# Anomaly count as crimson bars
plt.bar(
    ds_sub.index,
    ds_sub["anomaly_count"],
    color="crimson",
    label="Anomaly Count",
    alpha=0.8
)

# Anomaly rate on secondary y-axis
ax1 = plt.gca()
ax2 = ax1.twinx()
ax2.plot(
    ds_sub.index,
    ds_sub["anomaly_rate"],
    color="navy",
    marker="o",
    linestyle="-",
    label="Anomaly Rate (%)"
)

# 4) Tighten x-limits (optional half-day padding)
padding = pd.Timedelta(hours=12)
ax1.set_xlim(left_date - padding, right_date + padding)

# 5) Force ticks exactly at each date in ds_sub.index
ax1.set_xticks(ds_sub.index)

# 6) Label formatting: YYYY-MM-DD rotated 45°
ax1.set_xticklabels([d.strftime("%Y-%m-%d") for d in ds_sub.index], rotation=45)

# 7) Final labels, title, and combined legend
ax1.set_xlabel("Date")
ax1.set_ylabel("Number of Events / Anomalies")
ax2.set_ylabel("Anomaly Rate (%)")
plt.title("Daily Firewall Events & Anomaly Rate")

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
plt.legend(h1 + h2, l1 + l2, loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
# Extract the hour from DATE_TIME
df["HOUR_ONLY"] = df["DATE_TIME"].dt.hour

# Count anomalies vs. total by hour
hourly_total   = df.groupby("HOUR_ONLY").size()
hourly_anomaly = df.groupby("HOUR_ONLY")["ANOMALY_BIN"].sum()
hourly_rate    = (hourly_anomaly / hourly_total) * 100.0

# Plot a dual‐axis bar + line:
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.bar(hourly_total.index - 0.2, hourly_total.values, width=0.4, color="lightgray", label="Total Events")
ax1.bar(hourly_anomaly.index + 0.2, hourly_anomaly.values, width=0.4, color="crimson", label="Anomalies")

ax2 = ax1.twinx()
ax2.plot(hourly_rate.index, hourly_rate.values, color="navy", marker="o", label="Anomaly Rate (%)")

ax1.set_xlabel("Hour of Day")
ax1.set_ylabel("Event Count / Anomaly Count")
ax2.set_ylabel("Anomaly Rate (%)")
plt.title("Firewall Events & Anomaly Rate by Hour of Day")

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper right")

plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()
